##  Z-Score Normalization (Standardization)

### 🔹 Definition
Z-score normalization scales data so that each feature has:
- **Mean = 0**
- **Standard Deviation = 1**

It transforms values based on how far they are from the mean.

---

### 🔹 Formula

$$
z = \frac{x - \mu}{\sigma}
$$

Where:
- $x$ = original value  
- $\mu$ = mean of feature  
- $\sigma$ = standard deviation  

---

### 🔹 Advantages
- Removes scale differences between features  
- Centers data around zero → useful for many ML algorithms  
- Works well for normally distributed data  
- Improves convergence in optimization (e.g., gradient descent)

---

### 🔹 Disadvantages
- Not bounded (values not in a fixed range)  
- Sensitive to outliers  
- Assumes data is roughly normally distributed  

---

### 🔹 Notes
- Used in StandardScaler (scikit-learn)  
- Uses **population standard deviation (`ddof=0`)**

--------------------------------------------------------------------------------

## Why Z-Score is Sensitive to Outliers

### 🔹 Formula
$$
z = \frac{x - \mu}{\sigma}
$$

---

### 🔹 Key Idea
Z-score depends on:
- Mean ($\mu$)
- Standard deviation ($\sigma$)

Both are **highly affected by outliers**.

---

### 🔹 What goes wrong with outliers

#### 1. Mean shifts
- Outliers pull the mean toward themselves  
- Example:  
  - Normal: [10, 12, 14] → mean = 12  
  - With outlier: [10, 12, 14, 100] → mean = 34  

---

#### 2. Standard deviation increases
- Outliers increase spread → $\sigma$ becomes large  
- This compresses normal data after scaling  

---

#### 3. Z-scores become misleading
- Normal points appear closer to mean than they actually are  
- Outliers dominate the scaling process  

---

### 🔹 Intuition
Z-score measures:
> "How many standard deviations away is a point?"

If $\mu$ and $\sigma$ are distorted → Z-score becomes unreliable  

---

### 🔹 Impact in ML
- Feature scaling becomes poor  
- Important variations get compressed  
- Model performance may degrade  

---

### 🔹 Alternative
- Use RobustScaler (median + IQR) for outlier-heavy data  

---

### 🔹 One-line answer (Interview)
Z-score is sensitive to outliers because it uses mean and standard deviation, both of which are heavily influenced by extreme values.

In [1]:
import pandas as pd
import numpy as np

In [2]:
class ZScoreScaler:
    def __init__(self):
        self.mean_ = None
        self.std_ = None
        self.columns_ = None

    def fit(self, df: pd.DataFrame):
        """
        learn mean and std from training data
        """

        # select only numeric columns
        self.columns_ = df.select_dtypes(include = [np.number]).columns

        # compute mean 
        self.mean_ = df[self.columns_].mean()

        # compute std(delta degree of freedom ddof = 0 -> matces sklearn)
        self.std_ = df[self.columns_].std(ddof=0)

        # avoid division by zero -> replace 0 std with 1
        self.std_.replace(0,1,inplace = True)

        return self #allow method chaining 

    def transform(self, df:pd.DataFrame) -> pd.DataFrame:
        """
        Apply Z-score normalization using learned statistics
        """
        if self.mean_ is None or self.std_ is None:
            raise ValueError("Call fit() before transform()")
        
        df = df.copy() #avoid modifying orginal data

        # apply formula: (x - mean) / std
        df[self.columns_] = (df[self.columns_] - self.mean_) / self.std_
        return df
    
    def fit_transform(self, df:pd.DataFrame) -> pd.DataFrame:
        """
        Fit + transform in one step 
        """
        return self.fit(df).transform(df)

In [5]:
# sample data
df = pd.DataFrame({
    "A": [10, 20, 30, 40],
    "B": [1, 2, 3, 4],
    "C": ["cat", "dog", "cat", "dog"]  # non-numeric → ignored
})

scaler = ZScoreScaler()

print("befor scaling")
print(df)
# fit + transform training data
df_scaled = scaler.fit_transform(df)

print(df_scaled)

befor scaling
    A  B    C
0  10  1  cat
1  20  2  dog
2  30  3  cat
3  40  4  dog
          A         B    C
0 -1.341641 -1.341641  cat
1 -0.447214 -0.447214  dog
2  0.447214  0.447214  cat
3  1.341641  1.341641  dog


In [7]:
# # training
# scaler = ZScoreScaler()
# # after you divide your dataset from 
# X_train_scaled = scaler.fit_transform(X_train)

# # testing (use SAME mean/std)
# X_test_scaled = scaler.transform(X_test)